# NeMo Customizer - Testing Customized Model (Refactored)

This notebook **tests** the customized model after it has been deployed to RHOAI's Single Serving Runtime (SSR).  
**Refactored version** with helper functions and smaller cells for easier reading and editing. It does **not** download, merge, or upload the model—those steps use Python scripts from your machine (with port-forwards) because privileged `oc` or cluster access may not be available from the notebook.

## Prerequisites
- Completed training using `customize-model.ipynb`
- **Model deployment done via scripts** (from your machine): download adapter → merge with base → upload to MinIO. See README or `customize-model.ipynb` Step 18.
- InferenceService/SSR is configured to use the customized model path in MinIO
- `env.donotcommit` has `INFERENCE_SERVICE_URL`, `INFERENCE_SERVICE_NAME`, and optionally `CUSTOMIZED_MODEL_NAME`

## Step 1: Install Required Packages

In [ ]:
%pip install requests python-dotenv boto3

## Step 2: Import Libraries and Configure URLs

In [ ]:
import os
import json
import requests
from pathlib import Path

# Load environment variables from env.donotcommit if it exists
try:
    from dotenv import load_dotenv
    env_donotcommit_path = Path.cwd() / "env.donotcommit"
    if env_donotcommit_path.exists():
        load_dotenv(env_donotcommit_path, override=False)
        print(f"✅ Loaded env.donotcommit from: {env_donotcommit_path}")
    else:
        print(f"ℹ️  env.donotcommit not found at: {env_donotcommit_path}")
        print(f"   Using environment variables from system/defaults")
except ImportError:
    print("⚠️  python-dotenv not installed - using system environment variables only")

# Namespace configuration
NMS_NAMESPACE = os.getenv("NMS_NAMESPACE", "anemo-rhoai")

# Service URLs (cluster-internal)
CUSTOMIZER_URL = f"http://nemocustomizer-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
DATASTORE_URL = f"http://nemodatastore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
ENTITY_STORE_URL = f"http://nemoentitystore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"

# API Keys and Tokens (from env.donotcommit)
NIM_SERVICE_ACCOUNT_TOKEN = os.getenv("NIM_SERVICE_ACCOUNT_TOKEN", "")

# Inference Service Configuration (for testing models)
INFERENCE_SERVICE_URL = os.getenv("INFERENCE_SERVICE_URL", "")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "")
CUSTOMIZED_MODEL_NAME = os.getenv("CUSTOMIZED_MODEL_NAME", "")

print(f"Namespace: {NMS_NAMESPACE}")
print(f"Inference Service URL: {INFERENCE_SERVICE_URL}")
print(f"Inference Service Name: {INFERENCE_SERVICE_NAME}")
print(f"Customized Model Name: {CUSTOMIZED_MODEL_NAME}")

## Step 2b: Helper Functions

Define reusable functions used later for inference URL detection, running prompts, and saving responses. Run this cell once after configuring URLs (Step 2).

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def resolve_customized_model_name(inference_service_url, customized_model_name_env):
    """
    Return customized model name from env or user input, or None.
    Prints warnings and instructions if INFERENCE_SERVICE_URL or name not set.
    """
    if not inference_service_url:
        print("⚠️  INFERENCE_SERVICE_URL not set in env.donotcommit")
        print("   To test the customized model:")
        print("   1. Use Python scripts to download/merge/upload model to MinIO (see README)")
        print("   2. Update InferenceService to point to MinIO path, or create new InferenceService")
        print("   3. Set INFERENCE_SERVICE_URL to point to the InferenceService")
        print("   4. Set CUSTOMIZED_MODEL_NAME to the name of the customized model")
        print("   Example: INFERENCE_SERVICE_URL=http://your-customized-inference-service:8000")
        return None
    name = customized_model_name_env or None
    if not name:
        print("⚠️  CUSTOMIZED_MODEL_NAME not set")
        print("   Please set CUSTOMIZED_MODEL_NAME in env.donotcommit")
        print("   Or enter it manually below")
        user_input = input("Enter customized model name (or press Enter to skip): ").strip()
        if user_input:
            name = user_input
    if not name:
        print("⚠️  Could not determine customized model name")
        print("   Please set CUSTOMIZED_MODEL_NAME in env.donotcommit")
        print("   Or ensure the customization job response contains 'output_model'")
        return None
    print(f"✅ Found customized model: {name}")
    return name


def get_inference_headers(nim_service_account_token):
    """Return headers dict for inference API (Content-Type + optional Bearer token)."""
    headers = {"Content-Type": "application/json"}
    if nim_service_account_token:
        headers["Authorization"] = f"Bearer {nim_service_account_token}"
    return headers


def get_potential_urls_and_model_names(inference_service_url, inference_service_name, namespace, customized_model_name):
    """Return (potential_urls, model_names_to_try) for URL detection."""
    potential_urls = [inference_service_url]
    if inference_service_name:
        if inference_service_url.startswith("https://"):
            internal_url = f"https://{inference_service_name}-predictor.{namespace}.svc.cluster.local:8443"
        else:
            internal_url = f"http://{inference_service_name}-predictor.{namespace}.svc.cluster.local:80"
        potential_urls.append(internal_url)

    model_names_to_try = []
    if inference_service_name:
        model_names_to_try.append(inference_service_name)
    model_names_to_try.append(customized_model_name)
    if "@" in customized_model_name:
        model_names_to_try.append(customized_model_name.split("@")[0])
    if "/" in customized_model_name:
        name_part = customized_model_name.split("/")[-1]
        model_names_to_try.append(name_part)
        if "@" in name_part:
            model_names_to_try.append(name_part.split("@")[0])
    model_names_to_try = list(dict.fromkeys(model_names_to_try))
    return potential_urls, model_names_to_try


def detect_working_inference_url(potential_urls, model_names_to_try, headers):
    """
    Try each URL + model name until one returns 200. Returns (working_url, working_model_name) or (None, None).
    """
    for test_url in potential_urls:
        for model_name in model_names_to_try:
            try:
                test_payload = {
                    "model": model_name,
                    "messages": [{"role": "user", "content": "test"}],
                    "max_tokens": 5,
                }
                response = requests.post(
                    f"{test_url}/v1/chat/completions",
                    headers=headers,
                    json=test_payload,
                    timeout=10,
                    verify=False,
                )
                if response.status_code == 200:
                    return test_url, model_name
                elif response.status_code == 503:
                    print(f"   ⚠️  Model '{model_name}' returned 503 (service unavailable)")
                elif response.status_code == 404:
                    print(f"   ⚠️  Model '{model_name}' not found (404)")
                else:
                    print(f"   ⚠️  Model '{model_name}' returned {response.status_code}: {response.text[:100]}")
            except requests.exceptions.SSLError as ssl_err:
                print(f"   ⚠️  SSL error with {test_url}: {str(ssl_err)[:80]}")
                break
            except Exception as e:
                print(f"   ⚠️  Error with model '{model_name}': {str(e)[:80]}")
                continue
    return None, None


def run_inference_prompts(working_url, working_model_name, prompts, headers, max_tokens=200, temperature=0.7):
    """Run each prompt against the inference API. Returns dict prompt -> response text (or None on error)."""
    results = {}
    for i, prompt in enumerate(prompts, 1):
        print(f"\n[{i}/{len(prompts)}] Prompt: {prompt[:50]}...")
        try:
            payload = {
                "model": working_model_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens,
                "temperature": temperature,
            }
            response = requests.post(
                f"{working_url}/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=60,
                verify=False,
            )
            if response.status_code == 200:
                result = response.json()
                if result.get("choices") and len(result["choices"]) > 0:
                    response_text = result["choices"][0]["message"]["content"]
                    results[prompt] = response_text
                    print(f"✅ Response received ({len(response_text)} chars)")
                    print(f"   Preview: {response_text[:100]}...")
                else:
                    print(f"⚠️  Unexpected response format: {result}")
                    results[prompt] = None
            else:
                print(f"❌ Error: {response.status_code} - {response.text[:100]}")
                results[prompt] = None
        except Exception as e:
            print(f"❌ Exception: {str(e)[:100]}")
            results[prompt] = None
    return results


def save_responses_to_files(responses, json_path, txt_path):
    """Save responses dict to JSON and TXT files (same format as base model for comparison)."""
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(responses, f, indent=2, ensure_ascii=False)
    with open(txt_path, "w", encoding="utf-8") as f:
        for prompt, answer in responses.items():
            f.write(f"Prompt: {prompt}\n")
            f.write(f"Response: {answer if answer is not None else '(no response)'}\n")
            f.write("-" * 60 + "\n")


def print_troubleshooting(customized_model_name, inference_service_name, namespace, inference_service_url):
    """Print troubleshooting steps when no working URL was found."""
    print("\n❌ No working URL found. Cannot test customized model.")
    print("\n💡 Troubleshooting steps:")
    print(f"   1. Check if InferenceService is running:")
    print(f"      oc get inferenceservice {inference_service_name} -n {namespace}")
    print(f"   2. Check if pods are ready:")
    print(f"      oc get pods -n {namespace} | grep {inference_service_name or 'inference'}")
    print(f"   3. Verify the customized model is deployed:")
    print(f"      oc describe inferenceservice {inference_service_name} -n {namespace} | grep -i model")
    print(f"   4. Check service logs:")
    print(f"      oc logs -n {namespace} -l serving.kserve.io/inferenceservice={inference_service_name} --tail=50")
    print(f"   5. The customized model name is: {customized_model_name}")
    print(f"   6. Service URL: {inference_service_url}")
    print("\n⚠️  Note: If you just created the InferenceService, it may take a few minutes")
    print("   for the customized model to load. Wait and try again.")


def print_comparison_summary(customized_responses, working_model_name=None):
    """Print comparison summary and analysis tips. working_model_name used only in error branch."""
    successful = {k: v for k, v in customized_responses.items() if v is not None}
    if successful:
        print(f"\n✅ Customized model returned {len(successful)} successful response(s)")
        print("\n📝 Responses:")
        for prompt, response in successful.items():
            print(f"\n{'='*60}\nPrompt: {prompt}\n{'-'*60}\nResponse:\n{response}\n{'='*60}")
        print("\n💡 Analysis:")
        print("   - Review the responses above to verify customization worked")
        print("   - Compare with base model responses (if available) to see differences")
        print("   - Customized model should show changes in behavior based on training data")
    else:
        print("\n⚠️  No successful responses to display")
        print("   Check the errors above and verify:")
        print(f"   1. InferenceService is running and accessible")
        print(f"   2. Model name is correct: {working_model_name or 'N/A'}")
        print("   3. Model is loaded and ready in the InferenceService")

## Model deployment (done by scripts, not this notebook)

Download, merge, and upload of the customized model are done **from your machine** using the Python scripts (they may use `oc` and port-forwards). The notebook cannot run privileged `oc` commands.

**From your machine:**
1. `python export_model_from_entity_store.py` (or `--job-id <id>`)
2. `python download_model_from_datastore.py --model-info model_info.json --output-dir ./downloaded_model`
3. `python merge_adapter_with_base.py --adapter-dir ./downloaded_model --output-dir ./merged_model`
4. Port-forward MinIO, then `MINIO_ENDPOINT=http://localhost:9000 python upload_model_to_minio.py --model-dir ./merged_model --target-path models/llama-3.2-1b-instruct-cust`
5. Update InferenceService to point to that MinIO path, then run the steps below to **test** the model.

## Step 18: Test Customized Model

Now let's test the customized model to verify that customization actually changed the model's behavior. We use the **exact same 3 prompts** as in `customize-model.ipynb` Step 12 (base model), and save responses in the **same file format** so you can compare side by side:

- **Base model** (from customize-model.ipynb Step 12): `base_model_responses.json`, `base_model_responses.txt`
- **Customized model** (this notebook): `customized_model_responses.json`, `customized_model_responses.txt`

In [ ]:
# Step 18.1: Check prerequisites and get customized model name (uses helper from Step 2b)
customized_model_name = resolve_customized_model_name(INFERENCE_SERVICE_URL, CUSTOMIZED_MODEL_NAME)


In [ ]:
# Step 18.2: Detect working InferenceService URL for customized model (uses helpers from Step 2b)

if not customized_model_name:
    print("⚠️  Skipping URL detection - customized model name not available")
    working_url = None
    working_model_name = None
else:
    potential_urls, model_names_to_try = get_potential_urls_and_model_names(
        INFERENCE_SERVICE_URL, INFERENCE_SERVICE_NAME, NMS_NAMESPACE, customized_model_name
    )
    headers = get_inference_headers(NIM_SERVICE_ACCOUNT_TOKEN)
    print("🔍 Detecting working InferenceService URL for customized model...")
    print("   Will try model names:", model_names_to_try)

    working_url = None
    working_model_name = None
    for test_url in potential_urls:
        if working_url:
            break
        print("\n📡 Testing URL:", test_url)
        working_url, working_model_name = detect_working_inference_url(
            [test_url], model_names_to_try, headers
        )
        if working_url:
            print("✅ Found working URL:", test_url)
            print("✅ Working model name:", working_model_name)

    if not working_url:
        print_troubleshooting(
            customized_model_name, INFERENCE_SERVICE_NAME, NMS_NAMESPACE, INFERENCE_SERVICE_URL
        )
    else:
        print("\n✅ Ready to test! Using URL:", working_url)
        print("   Model name:", working_model_name)

In [ ]:
# Step 18.3a: Run inference with test prompts (uses helpers from Step 2b)
working_url = globals().get("working_url")
working_model_name = globals().get("working_model_name")

if not working_url or not working_model_name:
    print("⚠️  Skipping customized model testing - no working URL or model name")
    print("   Run Step 18.2 first to detect a working InferenceService URL")
    customized_responses = {}
else:
    test_prompts = [
        "What personal data does Red Hat collect?",
        "How does Red Hat use my personal data?",
        "What are my privacy rights with Red Hat?",
    ]
    print("📝 Using", len(test_prompts), "test prompts")
    print("\nTesting customized model responses using:", working_url)
    print("Model name:", working_model_name)
    print("=" * 60)

    headers = get_inference_headers(NIM_SERVICE_ACCOUNT_TOKEN)
    customized_responses = run_inference_prompts(
        working_url, working_model_name, test_prompts, headers, max_tokens=200, temperature=0.7
    )

    print("\n" + "=" * 60)
    print(
        "✅ Testing complete! Got",
        sum(1 for v in customized_responses.values() if v is not None),
        "/",
        len(test_prompts),
        "successful responses",
    )

In [ ]:
# Step 18.3b: Save responses to files (same format as base model for comparison)
customized_responses = globals().get("customized_responses", {})

if customized_responses:
    customized_responses_file = Path("customized_model_responses.json")
    customized_responses_txt = Path("customized_model_responses.txt")
    save_responses_to_files(customized_responses, customized_responses_file, customized_responses_txt)
    print("📁 Full responses saved to:", customized_responses_file.absolute(), "and", customized_responses_txt.absolute())
    print("   Compare with base model: base_model_responses.json / base_model_responses.txt (from customize-model.ipynb Step 12)")

In [ ]:
# Step 18.4: Compare customized model responses with base model (uses helper from Step 2b)
customized_responses = globals().get("customized_responses", {})

if not customized_responses:
    print("⚠️  No customized model responses available for comparison")
    print("   Run Step 18.3 first to test the customized model")
else:
    print("📊 Response Comparison Summary")
    print("=" * 60)
    print_comparison_summary(customized_responses, working_model_name=globals().get("working_model_name"))

## Summary - Testing Phase

This notebook completes the **testing** part of the customization workflow (deployment is done by Python scripts from your machine, not this notebook):

1. ✅ InferenceService URL detection (same prompts as base model)
2. ✅ Customized model testing with the 3 Red Hat privacy prompts
3. ✅ Responses saved to `customized_model_responses.json` and `.txt` for comparison with base
4. ✅ Side-by-side comparison of base vs customized model responses

**Model deployment** (download → merge → upload to MinIO) is done via scripts; this notebook only **tests** the model once SSR points to the customized path.

### Next Steps

1. Compare `base_model_responses.json` with `customized_model_responses.json` to validate customization
2. Adjust training data or parameters if needed and retrain
3. Deploy to production if results are satisfactory